# Phase 2 — Use case C: AI manufacturing reporting
**Muhammad Ibrahim · M04-0226-000404 · DATA722 Generative AI & Strategic Forecasting**

**Live app:** `https://YOUR-APP.vercel.app` &nbsp;&nbsp; **Repo:** `https://github.com/YOUR-USER/mfg-assistant`

This notebook is the working record for the build. Sections:

1. What was built and why it is *not* an agent
2. The data, and every cleaning decision
3. The prompt (the centre of the system) and the safety layer around it
4. The golden set (defined from the Phase 1 metrics) and a check that the scorer itself works
5. Evaluation of the built system, prompt v1
6. A failure, and what I did about it (prompt v2)
7. Token and cost figures
8. Check of the deployed app
9. Honesty section

> **How to read the outputs.** Sections 1–4 are fully offline and were executed when this file was saved.
> Sections 5–8 call the Groq API / the live app. Cells that need a key print *skipped* unless `GROQ_API_KEY` is set;
> **the scores in the final version of this notebook are the ones from my own run, not placeholders.**

## 0. Setup

In [1]:
import os, sys, json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
import engine
from eval import harness
pd.set_option("display.max_colwidth", 100); pd.set_option("display.width", 170)
HAVE_KEY = bool(os.getenv("GROQ_API_KEY"))
print("model:", engine.MODEL, "| Groq key found:", HAVE_KEY)

model: openai/gpt-oss-120b | Groq key found: False


## 1. What was built

**Use case C (Phase 1, Q1):** plant and planning staff ask production questions in plain language and get the number with its breakdown.

**Design decision carried over from Phase 1 (Q4): AI, not agentic.** The steps are fixed and the system only reads data:

```
question ──► LLM writes ONE read-only SQL query as JSON ──► SQL guard ──► run on SQLite ──► app displays the numbers
                  │                                            │
                  └── or answerable=false → clean refusal      └── if the query errors: ONE repair call, then stop
```

Three things are deliberate:
* **Only the question and the table definition go to the model, never data rows** (Phase 1, Q6b).
* **The model never writes the numbers the user reads.** The app runs the query and renders the result, so the model cannot invent a figure; the worst it can do is run the *wrong query*, and the query is shown on every answer (Phase 1, Q7a).
* **The "loop" is bounded**: at most 2 model calls per question (initial + one repair). There is no tool choice and no open-ended planning, so cost and latency are predictable and the system is testable against a golden set.

The four user-facing functions: (1) ask in English / Urdu / Roman Urdu → number or breakdown, (2) auto bar chart, (3) refusal of questions the data cannot answer, (4) a fixed year-overview with no model involved; plus a thumbs feedback log (Phase 1, Q5a).

## 2. The data and every cleaning decision

In [2]:
raw_path = ROOT / "private" / "raw_CVS_PROD_EFFICIENCY-2018.csv"      # raw file is NOT deployed (contains real brand/plant names)
if raw_path.exists():
    raw = pd.read_csv(raw_path, encoding="latin-1")
    q = lambda col: (raw[col].astype(str).str.strip() == "?").sum()
    date_like = lambda col: raw[col].astype(str).str.match(r"^\d{1,2}-[A-Za-z]{3}-\d{2}$").sum()
    issues = pd.Series({
        "rows": len(raw),
        "columns (2 of them are unnamed stray cells)": raw.shape[1],
        "file encoding": "latin-1 (not UTF-8)",
        "QUANTITY == '?'": q("QUANTITY"),
        "Production Time == '?'": q("Production Time"),
        "FAULT_DETAIL == '?' (no stoppage logged)": q("FAULT_DETAIL"),
        "loss cells corrupted into dates, e.g. '1-Jan-26'": sum(date_like(c) for c in ["PLN_LOSS","ELC_LOSS","MEC_LOSS","ELE_LOSS","OPT_LOSS"]),
        "distinct SHIFT values": raw["SHIFT"].nunique(),
        "distinct YEAR values": raw["YEAR"].nunique(),
        "columns for scrap / rejects": 0,
    })
    display(issues.to_frame("raw file"))
else:
    print("raw file not present (kept out of the repo on purpose); cleaned file below.")

,raw file
rows,5000
columns (2 of them are unnamed stray cells),26
file encoding,latin-1 (not UTF-8)
QUANTITY == '?',719
Production Time == '?',720
FAULT_DETAIL == '?' (no stoppage logged),1640
"loss cells corrupted into dates, e.g. '1-Jan-26'",8
distinct SHIFT values,1
distinct YEAR values,1
columns for scrap / rejects,0


**Cleaning decisions** (all in `scripts/prepare_data.py`, each one a choice that can change an answer):

| Raw | Decision | Why / risk |
|---|---|---|
| `?` in QUANTITY, Production Time | → NULL | Not recorded. SQL `SUM` skips NULL. About 60% of these rows mention a line closure in the stoppage text, the rest are unexplained. |
| `?` in the 5 `*_LOSS` columns | → 0 | I read it as "no stoppage logged". **This is an assumption**, not documented in the file. |
| Loss cells like `1-Jan-26` | → NULL (8 cells) | Excel turned a number into a date; the real value is unrecoverable. |
| `LINE_NO` | new unique `line` id such as `P1000-L3` | Line numbers repeat across plants, so "line 2" is ambiguous in the raw data. |
| Loss column meanings | PLN=planned, ELC=electrical, MEC=mechanical, ELE=utilities, OPT=operational | PLN/ELC/MEC/OPT are clear from the fault text; **ELE = utilities (power, air, gas) is inferred** from texts such as "air compressor trip". |
| Brand / plant / company names | replaced by `BRAND-A…G`, plant codes only | Phase 1 promised synthetic or anonymised data in the prototype; the app is public. A leak check in the script fails the build if a real name survives. |
| `Rated Speed`, `MI_CONV_250ML`, `PACK` | **dropped** | Units of rated speed are not documented and vary within the same package, so efficiency / OEE cannot be computed honestly. The system refuses those questions. |

In [3]:
df = pd.read_csv(ROOT / "data" / "production.csv")
print(df.shape)
display(df.head(4).T)
cov = pd.DataFrame({"missing": df.isna().sum(), "pct": (100 * df.isna().mean()).round(1)})
display(cov[cov.missing > 0])
print("shifts:", df["shift"].unique().tolist(), "| years:", df["year"].unique().tolist(), "| lines:", df["line"].nunique(), "| plants:", df["plant"].nunique())

(5000, 23)


,0,1,2,3
record_id,1,2,3,4
date,2018-08-07,2018-08-29,2018-07-02,2018-03-17
year,2018,2018,2018,2018
month,8,8,7,3
month_name,Aug,Aug,Jul,Mar
shift,A,A,A,A
plant,7000,4000,4000,7000
line,P7000-L1,P4000-L1,P4000-CO2,P7000-L1
line_no,1,1,2,1
product,BRAND-F MANGO REGULAR 240 ML,BRAND-D 250 ML,BRAND-B PET 2250 ML,BRAND-F MANGO PET 1000 ML


,missing,pct
cases,719,14.4
run_hours,720,14.4
planned_loss_hrs,6,0.1
electrical_loss_hrs,2,0.0
fault_detail,1640,32.8


shifts: ['A'] | years: [2018] | lines: 16 | plants: 5


## 3. The prompt (centre of the system) and the safety layer

`engine.build_system_prompt()` builds the prompt from the same `SCHEMA` list that creates the table, so the prompt and the database cannot disagree.
Design choices: column dictionary with the *traps written out* (NULL cases, repeated line numbers, only shift A), explicit refusal rules, a JSON output contract, and 5 few-shot examples (3 answerable, 2 refusals).

In [4]:
p = engine.build_system_prompt()
print(f"{len(p):,} characters, roughly {len(p)//4:,} tokens (estimate; the real count is in the Groq usage field)\n")
print(p)

5,640 characters, roughly 1,410 tokens (estimate; the real count is in the Groq usage field)

You convert production questions from a beverage manufacturer into ONE read-only SQLite query.
You never see the data rows, only this table definition. Reply with a single JSON object and nothing else.

TABLE production  (5000 rows, dates 2018-01-01 to 2018-12-31, shift A only)
- record_id (INTEGER): unique id of one production record (a line running one product on one date)
- date (TEXT): production date, ISO format YYYY-MM-DD
- year (INTEGER): always 2018
- month (INTEGER): 1-12
- month_name (TEXT): Jan, Feb, ... Dec
- shift (TEXT): always 'A' - the only shift present in the data
- plant (INTEGER): plant code: 1000, 2000, 3000, 4000, 7000
- line (TEXT): unique line id such as 'P1000-L3'. USE THIS for any per-line question
- line_no (INTEGER): line number inside a plant. NOT unique across plants - never group by it alone
- product (TEXT): product name incl. pack, e.g. 'BRAND-C PET 500 ML'
- b

### The SQL guard
The model's SQL is untrusted. Three independent layers: (1) a text check (one statement, `SELECT`/`WITH` only, no comments, forbidden keywords, ignoring text inside quotes so `LIKE '%REPLACE%'` still works),
(2) a SQLite *authorizer* that only permits reading the `production` table, (3) a 3-second timeout and a 200-row cap.

In [5]:
attacks = [
  "DROP TABLE production",
  "SELECT 1; DELETE FROM production",
  "UPDATE production SET cases = 0",
  "SELECT * FROM sqlite_master",
  "SELECT * FROM production -- ignore the rest",
  "PRAGMA table_info(production)",
  "WITH x AS (SELECT 1) DELETE FROM production",
  "SELECT COUNT(*) FROM production WHERE fault_detail LIKE '%REPLACE%'",   # legitimate: keyword inside quotes
]
out = []
for s in attacks:
    try:
        engine.run_sql(engine.validate_sql(s)); res = "ALLOWED (legitimate)" if "LIKE" in s else "!! ALLOWED"
    except (engine.SQLRejected, engine.sqlite3.Error) as e:
        res = f"blocked: {e}"
    out.append({"sql": s, "result": res})
pd.DataFrame(out)

,sql,result
0,DROP TABLE production,blocked: only SELECT queries are allowed
1,SELECT 1; DELETE FROM production,blocked: only one statement is allowed
2,UPDATE production SET cases = 0,blocked: only SELECT queries are allowed
3,SELECT * FROM sqlite_master,blocked: access to sqlite_master.type is prohibited
4,SELECT * FROM production -- ignore the rest,blocked: comments are not allowed
5,PRAGMA table_info(production),blocked: only SELECT queries are allowed
6,WITH x AS (SELECT 1) DELETE FROM production,blocked: keyword 'DELETE' is not allowed
7,SELECT COUNT(*) FROM production WHERE fault_detail LIKE '%REPLACE%',ALLOWED (legitimate)


## 4. The golden set and a check of the scorer

Phase 1 (Q7) defined the metrics: execution accuracy, correct breakdown rate, correct refusal rate, confident-wrong rate, latency and cost.
I wrote **39 cases**: 15 single-number, 12 breakdown, 12 must-refuse. They include the traps I know are in the data (ambiguous "line 2", relative dates, free-text spelling variants, Urdu and Roman Urdu, scrap/OEE/other-shift/other-year, a `DROP TABLE` injection, a prompt-leak request).

**The expected answers are not typed by hand.** Each case has a reference SQL that I run on the same database, so the golden answer cannot drift from the data.

In [6]:
gold = harness.load_golden()
rows = []
for c in gold:
    if c["kind"] == "refuse":
        exp = "REFUSE"
    else:
        r = harness.expected(c)
        exp = r[0][0] if c["kind"] == "scalar" else f"{len(r)} rows, e.g. {r[:2]}"
    rows.append({"id": c["id"], "kind": c["kind"], "category": c["category"], "question": c["question"], "golden answer": exp})
gdf = pd.DataFrame(rows); display(gdf)
print(gdf.kind.value_counts().to_dict())

,id,kind,category,question,golden answer
0,S01,scalar,total,How many cases were produced in total in 2018?,34960413.0
1,S02,scalar,time filter,How many cases did we produce in March?,3259954.0
2,S03,scalar,time filter,Total cases for plant 1000 in the second quarter,6607951.0
3,S04,scalar,line filter,How many cases did line P1000-L3 make in June?,422960.0
4,S05,scalar,count,How many production records are there for plant 3000?,757
5,S06,scalar,downtime,What are the total downtime hours across all plants?,8416.5
6,S07,scalar,downtime,How many hours were lost to mechanical faults on line P1000-L4?,60.0
7,S08,scalar,text search,How many records mention a changeover in the stoppage log?,1195
8,S09,scalar,product filter,How many cases of BRAND-C were produced in the third quarter?,2367689.0
9,S10,scalar,product filter,Total cases of can products,3760500.0


{'scalar': 15, 'table': 12, 'refuse': 12}


### Does the scorer work? (harness check, **not** system scores)
Before trusting any score I check the scorer against two fake models: an *oracle* that writes the reference SQL (must score 100%) and a *saboteur* that is confidently wrong and never refuses (must be caught).

In [7]:
sys.path.insert(0, str(ROOT / "tests"))
from test_offline import oracle, sabotaged
print("=== ORACLE (should be perfect) ===");   harness.print_summary(harness.summarise(harness.run(llm=oracle, verbose=False)))
print("\n=== SABOTEUR (should be caught) ==="); harness.print_summary(harness.summarise(harness.run(llm=sabotaged, verbose=False)))

=== ORACLE (should be perfect) ===



Cases: 39   model: openai/gpt-oss-120b   prompt v2
Execution accuracy (single numbers): 15/15 = 100%
Correct breakdown rate (tables):     12/12 = 100%
Correct refusal rate:                12/12 = 100%
False refusals (answerable refused): 0/27 = 0%
Errors:                              0/39 = 0%
CONFIDENT-WRONG rate:                0/39 = 0%
Latency ms  mean 1  p50 0  p95 3
Tokens/question  in 1400  out 120
Cost  total $0.011   per question $0.000282

=== SABOTEUR (should be caught) ===



Cases: 39   model: openai/gpt-oss-120b   prompt v2
Execution accuracy (single numbers): 10/15 = 67%
Correct breakdown rate (tables):     11/12 = 92%
Correct refusal rate:                0/12 = 0%
False refusals (answerable refused): 0/27 = 0%
Errors:                              0/39 = 0%
CONFIDENT-WRONG rate:                18/39 = 46%
Latency ms  mean 0  p50 0  p95 2
Tokens/question  in 1400  out 120
Cost  total $0.011   per question $0.000282


## 5. Evaluation of the built system — prompt v1
Runs all 39 cases through the real pipeline (`engine.answer`) with the real model.
On Groq's free tier the token-per-minute limit is tight (~1.5k tokens per question), so `PAUSE` spaces the calls; set it to 0 on a paid key.

In [8]:
PAUSE = 10                                   # seconds between questions (free tier). 0 on a paid key.
os.environ["GROQ_MAX_WAIT"] = "60"           # wait out 429 rate-limit responses instead of failing the case

def run_eval(version):
    engine.PROMPT_VERSION = version
    res = harness.run(pause_s=PAUSE)
    summ = harness.summarise(res); harness.print_summary(summ)
    json.dump({"summary": summ, "results": res}, open(ROOT / "eval" / f"results_v{version}.json", "w"), indent=1, ensure_ascii=False)
    return res, summ

res_v1 = summ_v1 = None
if HAVE_KEY:
    res_v1, summ_v1 = run_eval("1")
else:
    print("SKIPPED: set GROQ_API_KEY and re-run this cell to produce the real scores.")

SKIPPED: set GROQ_API_KEY and re-run this cell to produce the real scores.


In [9]:
if res_v1:
    r1 = pd.DataFrame(res_v1)
    display(r1.groupby("kind")["pass"].agg(passed="sum", total="count"))
    display(r1[["id","question","status","pass"]].assign(result=lambda d: d["pass"].map({True:"pass", False:"FAIL"})).drop(columns="pass"))

## 6. A failure I found, and what I did about it

In [10]:
if res_v1:
    fails = pd.DataFrame([r for r in res_v1 if not r["pass"]])
    display(fails[["id","category","question","status","why"]] if len(fails) else "no failures")
    for _, f in fails.iterrows():
        print(f"--- {f['id']}: {f['question']}\n    SQL: {f['sql']}\n    why: {f['why']}\n")
else:
    print("SKIPPED until section 5 has run.")

SKIPPED until section 5 has run.


### Failure log  *(fill from the table above — write what actually happened, not what I expected)*

**Failure #1 — case ID:** `[e.g. S08]`
* **What the system did:** `[paste the SQL and the number it returned]`
* **What was right:** `[golden answer and the reference SQL]`
* **Root cause:** `[e.g. the model matched '%CHANGE OVER%' but the free text also spells it 'CHANGEOVER', so 7 records were missed. The answer carried no warning.]`
* **Why the guard did not catch it:** the query was valid, read-only and plausible; only a person comparing against the data could notice.
* **Fix tried:** prompt v2 adds rule 12 (OR together spelling variants of a keyword).
* **Result of the fix:** `[v1 → v2 score on this case and on the whole set, from the cell below]`
* **What the fix does NOT solve:** it teaches the model about *one* known pattern. It will not know about spelling variants I have not seen. And the golden case S08 was written by me, with the same knowledge as the fix, so a pass on it overstates how well this generalises.

*Pre-registered hypotheses (written before running):* I expected trouble on S08 (free-text variants), B11 ("line 2" across plants), S13 ("last month" has no calendar to anchor on), and the Urdu-script case S15. If the real failures differ, the real ones are what belong above.

In [11]:
res_v2 = summ_v2 = None
if HAVE_KEY:
    res_v2, summ_v2 = run_eval("2")
else:
    print("SKIPPED: needs GROQ_API_KEY.")

SKIPPED: needs GROQ_API_KEY.


In [12]:
if res_v1 and res_v2:
    keys = ["execution_accuracy","correct_breakdown_rate","correct_refusal_rate","false_refusals","errors","confident_wrong"]
    cmp = pd.DataFrame({"v1": {k: "%d/%d" % summ_v1[k] for k in keys}, "v2": {k: "%d/%d" % summ_v2[k] for k in keys}})
    display(cmp)
    a, b = pd.DataFrame(res_v1).set_index("id")["pass"], pd.DataFrame(res_v2).set_index("id")["pass"]
    print("fixed by v2 :", a[(~a) & b].index.tolist())
    print("broken by v2:", a[a & (~b)].index.tolist())
    print("still failing:", b[~b].index.tolist())

## 7. Token and cost figures
Pricing used (Groq list prices, USD per 1M tokens, checked October 2026): `openai/gpt-oss-120b` input 0.15, output 0.60. **Verify on console.groq.com before quoting.**
Cached-token discounts are ignored, so the figures are a slight over-estimate.

In [13]:
final = res_v2 or res_v1
if final:
    d = pd.DataFrame(final)
    answered = d[d.status != "error"]
    print(f"model {engine.MODEL}, {len(d)} questions")
    print(f"mean tokens/question   in {d.prompt_tokens.mean():,.0f}   out {d.completion_tokens.mean():,.0f}")
    print(f"mean cost/question     ${d.cost_usd.mean():.5f}      total for this run ${d.cost_usd.sum():.4f}")
    print(f"latency ms             mean {d.latency_ms.mean():,.0f}   p50 {d.latency_ms.median():,.0f}   p95 {d.latency_ms.quantile(.95):,.0f}")
    print(f"questions needing the repair call: {int(d.repaired.sum())}")
    per_q = d.cost_usd.mean()
    display(pd.DataFrame({"questions / month": [1_000, 10_000, 100_000],
                          "LLM cost (USD)": [round(n * per_q, 2) for n in (1_000, 10_000, 100_000)]}))
else:
    print("SKIPPED: needs a real run. Pre-run estimate for planning only: ~1,400 tokens in + ~150-400 out per question,")
    print("i.e. roughly $0.0004-$0.0005 per question at gpt-oss-120b list price. Replace with measured figures.")

SKIPPED: needs a real run. Pre-run estimate for planning only: ~1,400 tokens in + ~150-400 out per question,
i.e. roughly $0.0004-$0.0005 per question at gpt-oss-120b list price. Replace with measured figures.


## 8. Check of the deployed app
Calls the live Vercel URL (no key needed on my side: the server holds it). Set `APP_URL` to the deployed address.

In [14]:
import httpx, time
APP_URL = "https://YOUR-APP.vercel.app"
if "YOUR-APP" in APP_URL:
    print("SKIPPED: set APP_URL to the deployed address.")
else:
    print(httpx.get(APP_URL + "/api/health", timeout=20).json())
    for q in ["How many cases did we produce in March?", "What was the scrap rate on line P1000-L2?"]:
        t = time.time(); r = httpx.post(APP_URL + "/api/ask", json={"question": q}, timeout=60).json()
        print(f"{time.time()-t:4.1f}s  {r['status']:8s} {q}  ->  {r.get('rows') or r.get('reason')}")

SKIPPED: set APP_URL to the deployed address.


## 9. Honesty section

**What this is.** A question-to-SQL assistant over a 5,000-row extract of one year (2018) of one plant group's production records, with brand and plant names anonymised. It is a prototype of Phase 1 use case C, not the SAP HANA system described there.

### What the system gets wrong
* **It can run the wrong query and look right.** The arithmetic is always correct (the app runs the SQL), but the question answered may not be the one asked. The only protection is that the query is shown; a plant manager who does not read SQL cannot check it. My confident-wrong rate on the golden set was **[FILL: x/39 = y%]**; every one of those had a number on screen and no warning.
* **Free text is the weakest spot.** Stoppage reasons are typed by people with spelling variants, and anonymisation replaced a utility name in that text, so keyword counts ("power failure") will not match what the real data would give. Failure found in testing: **[FILL: case ID, what happened, what v2 changed, v1→v2 score]**. The fix teaches one pattern I already knew about; my golden case for it was written by me, so the pass overstates how well it generalises.
* **"Line 2" is ambiguous** (it exists in several plants) and **"last month" has no calendar** (the data stops at 31 Dec 2018). The system states its interpretation, but it is guessing.
* **Data assumptions I made, not the file:** `?` in loss columns means "no stoppage" (set to 0); the `ELE` column is utilities (inferred); 14.4% of records have no quantity and are silently excluded from totals (the app prints this on every answer, but a total is still a lower bound). Eight loss cells corrupted into dates were nulled.
* **Urdu and Roman Urdu** were tested on **[FILL: n]** questions only. I cannot claim a rate.

### What I would not yet put in front of a customer
* Anything where a wrong number drives a shift or management decision, without a second check of the SQL by someone who reads it.
* **No access control.** Phase 1 says access follows each user's SAP role; here anyone with the link sees everything.
* **Not tested against real SAP data, real load, or hostile users.** Rate limits and the answer cache live in memory per server instance, so on Vercel they are best-effort, not a guarantee. The query log is in `/tmp` and disappears; it is also printed to Vercel's logs.
* **Refusal depends on the model following written rules.** There is no separate classifier. A persistent user could probably get an approximate "efficiency" figure by asking for the ingredients of one. Scrap, OEE, other shifts and other years are refused because the data does not contain them, not because they are blocked.
* **The golden set is small and mine.** 39 cases written by the person who built the prompt. It measures regressions, not real-world accuracy; I would want 200+ questions from actual plant staff.
* Data left Pakistan: the question and table definition go to Groq's API (US-hosted). Phase 1 flagged this as the condition that would turn the verdict into No-go if localisation were required. The data rows never leave the server.

### What it cost
* **Model:** `openai/gpt-oss-120b` on Groq, list price $0.15 in / $0.60 out per million tokens. Measured: **[FILL: mean tokens in/out per question, cost per question]**; the full 39-question eval cost **[FILL: $]**. Projected **[FILL: $]** per 10,000 questions a month. Cached-token discounts are ignored.
* **Latency:** mean **[FILL] ms**, p95 **[FILL] ms** per question.
* **Hosting:** Vercel Hobby plan and Groq free tier: $0 while demo traffic stays small. The free tier's tokens-per-minute limit is the real constraint: a handful of simultaneous markers could hit it. I added a per-IP limit, a daily cap, and an answer cache for repeated questions to protect the budget.
* **My time:** **[FILL: hours]**, plus AI assistance in writing the code.